In [2]:
import pandas as pd

df = pd.read_csv('cleaned_dataset.csv')
raw = pd.read_csv('train.csv')      
df.shape                         

(891, 12)

### Score the 6 DAMA dimensions

In [3]:
scores = {}

# Completeness: % of cells that are not null
scores['Completeness'] = round((1 - df.isnull().sum().sum() / df.size) * 100, 1)
scores['Completeness (real Cabin values)'] = round((df['Cabin'] != 'Unknown').mean() * 100, 1)

# Uniqueness: % of PassengerId values that are not duplicates
scores['Uniqueness'] = round((1 - df['PassengerId'].duplicated().mean()) * 100, 1)

# Validity: % of rows where every value is allowed
valid = (df['Sex'].isin(['Male', 'Female']) & df['Pclass'].isin([1, 2, 3]) & df['Survived'].isin([0, 1])
         & df['Age'].between(0, 100) & (df['Fare'] >= 0)
         & df['Embarked'].isin(['Southampton', 'Cherbourg', 'Queenstown']))
scores['Validity'] = round(valid.mean() * 100, 1)

# Accuracy (proxy): % of Age and Fare values that are original, not imputed or capped
original = (raw['Age'].notnull().sum() + (raw['Fare'] == df['Fare']).sum()) / (2 * len(raw))
scores['Accuracy (proxy)'] = round(original * 100, 1)

# Consistency: title in Name agrees with Sex (Mr/Master = Male, Mrs/Miss/Ms/Mlle/Mme = Female)
title = df['Name'].str.extract(r',\s*([^.]+)\.')[0]
bad = ((title.isin(['Mr', 'Master']) & (df['Sex'] != 'Male')) |
       (title.isin(['Mrs', 'Miss', 'Ms', 'Mlle', 'Mme']) & (df['Sex'] != 'Female')))
scores['Consistency'] = round((1 - bad.mean()) * 100, 1)

for k, v in scores.items():
    print(f'{k}: {v}%')

Completeness: 100.0%
Completeness (real Cabin values): 23.5%
Uniqueness: 100.0%
Validity: 100.0%
Accuracy (proxy): 83.6%
Consistency: 100.0%


### Write the Great Expectations suite (8 expectations)

In [4]:
import great_expectations as gx
import great_expectations.expectations as gxe

context = gx.get_context()
source = context.data_sources.add_pandas('titanic_source')
asset = source.add_dataframe_asset(name='titanic')
batch_def = asset.add_batch_definition_whole_dataframe('whole_df')

suite = context.suites.add(gx.ExpectationSuite(name='titanic_suite'))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='Age'))
suite.add_expectation(gxe.ExpectColumnValuesToBeUnique(column='PassengerId'))
suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column='Sex', value_set=['Male', 'Female']))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column='Age', min_value=0, max_value=100))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column='Fare', min_value=0))
suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column='Pclass', value_set=[1, 2, 3]))
suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column='Embarked', value_set=['Southampton', 'Cherbourg', 'Queenstown']))
suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column='Survived', value_set=[0, 1]))

validation = context.validation_definitions.add(
    gx.ValidationDefinition(name='titanic_validation', data=batch_def, suite=suite))

In [5]:
def run(data):
    return validation.run(batch_parameters={'dataframe': data})

def show(result):
    print('Overall passed:', result.success)
    for r in result.results:
        cfg = r.expectation_config
        print(' ', 'PASS' if r.success else 'FAIL', cfg.type, cfg.kwargs.get('column'))

In [6]:
show(run(df))

Calculating Metrics:   0%|          | 0/55 [00:00<?, ?it/s]

Overall passed: True
  PASS expect_column_values_to_not_be_null Age
  PASS expect_column_values_to_be_between Age
  PASS expect_column_values_to_be_unique PassengerId
  PASS expect_column_values_to_be_in_set Sex
  PASS expect_column_values_to_be_between Fare
  PASS expect_column_values_to_be_in_set Pclass
  PASS expect_column_values_to_be_in_set Embarked
  PASS expect_column_values_to_be_in_set Survived


### Inject 3 errors and confirm GX catches them

In [7]:
bad = df.copy()
bad.loc[5, 'Age'] = None                                  # error 1: missing value
bad.loc[10, 'Sex'] = 'Unknown'                            # error 2: invalid category
bad.loc[1, 'PassengerId'] = bad.loc[0, 'PassengerId']     # error 3: duplicate ID

result = run(bad)
show(result)

failed = [r for r in result.results if not r.success]
print(len(failed), 'of', len(result.results), 'failed')
for r in failed:
    print(r.expectation_config.type, r.expectation_config.kwargs.get('column'), '| bad rows:', r.result.get('unexpected_count'))

Calculating Metrics:   0%|          | 0/55 [00:00<?, ?it/s]

Overall passed: False
  FAIL expect_column_values_to_not_be_null Age
  PASS expect_column_values_to_be_between Age
  FAIL expect_column_values_to_be_unique PassengerId
  FAIL expect_column_values_to_be_in_set Sex
  PASS expect_column_values_to_be_between Fare
  PASS expect_column_values_to_be_in_set Pclass
  PASS expect_column_values_to_be_in_set Embarked
  PASS expect_column_values_to_be_in_set Survived
3 of 8 failed
expect_column_values_to_not_be_null Age | bad rows: 1
expect_column_values_to_be_unique PassengerId | bad rows: 2
expect_column_values_to_be_in_set Sex | bad rows: 1


### Data contract YAML

In [8]:
import yaml

contract = """
dataset: titanic_passengers
version: 1.0.0
description: Cleaned Titanic passenger data used by the BI dashboard
owner:
  producer: Benjamin
  contact: add-your-email-here
consumer:
  name: BI dashboard
  use: survival and fare analysis by class, sex, age band and port
delivery:
  file: cleaned_dataset.csv
  format: csv
  refresh: on change
schema:
  - {name: PassengerId, type: integer, required: true, unique: true, pii: none}
  - {name: Survived, type: integer, required: true, allowed: [0, 1], pii: none}
  - {name: Pclass, type: integer, required: true, allowed: [1, 2, 3], pii: none}
  - {name: Name, type: string, required: true, pii: direct_identifier, handling: tokenise}
  - {name: Sex, type: string, required: true, allowed: [Male, Female], pii: quasi_identifier, handling: keep_restricted}
  - {name: Age, type: float, required: true, min: 0, max: 100, pii: quasi_identifier, handling: generalise_to_age_band}
  - {name: SibSp, type: integer, required: true, min: 0, pii: none}
  - {name: Parch, type: integer, required: true, min: 0, pii: none}
  - {name: Ticket, type: string, required: true, pii: indirect_identifier, handling: tokenise}
  - {name: Fare, type: float, required: true, min: 0, pii: none}
  - {name: Cabin, type: string, required: true, pii: quasi_identifier, handling: generalise_to_deck}
  - {name: Embarked, type: string, required: true, allowed: [Southampton, Cherbourg, Queenstown], pii: none}
quality:
  row_count: {min: 800, max: 1000}
  completeness_min_pct: 99
  duplicate_passenger_id: 0
  freshness: not enforced (static historical dataset; add a loaded_at column if it is refreshed regularly)
  validated_with: great_expectations suite titanic_suite (8 expectations)
change_policy:
  breaking_changes: renaming or removing a column, changing a type, or changing allowed values
  notice_days: 14
  version_rule: breaking change raises the major version
"""

c = yaml.safe_load(contract)          # confirms the YAML is valid
print(c['dataset'], c['version'], '| producer:', c['owner']['producer'], '| consumer:', c['consumer']['name'])
open('data_contract.yaml', 'w').write(contract.strip() + '\n')

titanic_passengers 1.0.0 | producer: Benjamin | consumer: BI dashboard


1894

### Apply the masking

In [9]:
import hmac, hashlib, os

KEY = os.environ.get('TOKEN_KEY', 'change-this-secret').encode()   # keep the real key outside the repo

def token(value):
    return hmac.new(KEY, str(value).encode(), hashlib.sha256).hexdigest()[:12]

masked = df.copy()
masked['name_token'] = masked['Name'].map(token)
masked['ticket_token'] = masked['Ticket'].map(token)
masked['age_band'] = pd.cut(masked['Age'], bins=[0, 12, 17, 59, 100],
                            labels=['0-12', '13-17', '18-59', '60+'], include_lowest=True)
masked['deck'] = masked['Cabin'].str[0].where(masked['Cabin'] != 'Unknown', 'Unknown')

masked = masked.drop(columns=['Name', 'Ticket', 'Age', 'Cabin'])
masked.to_csv('cleaned_dataset_masked.csv', index=False)

masked.head(3)

,PassengerId,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked,name_token,ticket_token,age_band,deck
0,1,0,3,Male,1,0,7.2500,Southampton,68d653dc018e,e7c257b1a6a6,18-59,Unknown
1,2,1,1,Female,1,0,65.6344,Cherbourg,ca578c32e9de,935e78cf856c,18-59,C
2,3,1,3,Female,0,0,7.9250,Southampton,9b6671cd61e5,2d6ed62a5848,18-59,Unknown


In [10]:
print(masked['age_band'].value_counts().to_dict())            # {'18-59': 752, '0-12': 69, '13-17': 44, '60+': 26}
print(masked['name_token'].nunique(), masked['ticket_token'].nunique(), df['Ticket'].nunique())   # 891 681 681

{'18-59': 752, '0-12': 69, '13-17': 44, '60+': 26}
891 681 681
